## Bagging Classification

In [1]:
import numpy as np
import pandas as pd

In [5]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [39]:
# 1. Load the dataset
iris= load_iris()

df = pd.DataFrame(iris.data, columns= iris.feature_names)

df['target'] = iris.target
df.head()

# Keep only two classes and two spe
df = df[df['target'] != 0] [['sepal width (cm)', 'petal length (cm)', 'target']]

df

,sepal width (cm),petal length (cm),target
50,3.2,4.7,1
51,3.2,4.5,1
52,3.1,4.9,1
53,2.3,4.0,1
54,2.8,4.6,1
...,...,...,...
145,3.0,5.2,2
146,2.5,5.0,2
147,3.0,5.2,2
148,3.4,5.4,2


In [41]:
# 2. Separate the features and the target
X = df.drop('target', axis=1)
y = df['target']


In [42]:
# 3. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size =0.2, random_state=42, stratify=y)

In [53]:
# 6. CREATE BOOTSTRAP SAMPLES AND TRAIN MULTIPLE DECISION TREES

models = []
n_estimators = 10

for i in range(n_estimators):
    df_bag = pd.concat([X_train, y_train], axis=1).sample(len(X_train), replace=True, random_state=i)

    # Separating features and target
    X_bag = df_bag[['sepal width (cm)', 'petal length (cm)']]
    y_bag = df_bag['target']

    # Create Decision Tree
    model = DecisionTreeClassifier(random_state=i)

    # Train Decision Tree
    model.fit(X_bag, y_bag)

    # Store the model
    models.append(model)

models

[DecisionTreeClassifier(random_state=0),
 DecisionTreeClassifier(random_state=1),
 DecisionTreeClassifier(random_state=2),
 DecisionTreeClassifier(random_state=3),
 DecisionTreeClassifier(random_state=4),
 DecisionTreeClassifier(random_state=5),
 DecisionTreeClassifier(random_state=6),
 DecisionTreeClassifier(random_state=7),
 DecisionTreeClassifier(random_state=8),
 DecisionTreeClassifier(random_state=9)]

In [54]:
# 7. GET PREDICTIONS FROM EVERY TREE

predictions = []

for model in models:
    y_pred = model.predict(X_test)

    predictions.append(y_pred)

predictions = np.array(predictions)

predictions

array([[2, 2, 2, 2, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 1, 2, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1],
       [2, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 2, 2, 2, 1, 1, 2, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2],
       [2, 1, 2, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2],
       [2, 2, 2, 2, 1, 1, 2, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2]])

In [57]:
# 8. Majority Voting
final_predictions =[]

for i in range(len(X_test)):
    # Predictions made by all tree
    votes = predictions[:,i]

    # Most common class
    final_prediction = np.bincount(votes).argmax()

    final_predictions.append(final_prediction)

final_predictions = np.array(final_predictions)

final_predictions

array([2, 2, 2, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2])

In [58]:
# 9. EVALUATE BAGGING MODEL

accuracy = accuracy_score(y_test, final_predictions)

print("Number of trees:", n_estimators)

print("Bagging Accuracy:", accuracy)

Number of trees: 10
Bagging Accuracy: 0.85


# The important part to understand

## The entire algorithm is these three stages:
                 
                 TRAINING DATA
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
      Bootstrap   Bootstrap   Bootstrap
       Sample 1    Sample 2    Sample 3
          ↓           ↓           ↓
        Tree 1      Tree 2      Tree 3
          ↓           ↓           ↓
          └───────────┼───────────┘
                      │
                 NEW DATA
                      ↓
             All trees predict
                      ↓
              [1, 2, 1, 1, 2...]
                      ↓
               MAJORITY VOTE
                      ↓
                FINAL CLASS

In [59]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score


# ============================================
# 1. LOAD IRIS DATASET
# ============================================

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df['target'] = iris.target

df.head()


# ============================================
# 2. SEPARATE FEATURES AND TARGET
# ============================================

X = df.drop('target', axis=1)

y = df['target']


# ============================================
# 3. TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================
# 4. CREATE BAGGING CLASSIFIER
# ============================================

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=10,
    bootstrap=True,
    random_state=42
)


# ============================================
# 5. TRAIN BAGGING MODEL
# ============================================

bagging_model.fit(
    X_train,
    y_train
)


# ============================================
# 6. MAKE PREDICTIONS
# ============================================

y_pred = bagging_model.predict(
    X_test
)


# ============================================
# 7. EVALUATE MODEL
# ============================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Number of trees:", bagging_model.n_estimators)
print("Bagging Accuracy:", accuracy)

Number of trees: 10
Bagging Accuracy: 0.9666666666666667
